[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_11_advantage_actor_critic_complete.ipynb)

<div style="text-align:center">
    <h1>
        Advantage Actor-Critic (A2C)
    </h1>
</div>

<br><br>

<div style="text-align:center">
In this notebook we are going to combine temporal difference learning (TD) with policy gradient methods. The resulting algorithm is called Advantage Actor-Critic (A2C) and uses a one-step estimate of the return to update the policy:
</div>

\begin{equation}
\hat G_t = R_{t+1} + \gamma v(S_{t+1}|w)
\end{equation}


<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_policy_network, plot_stats, seed_everything


## Import the necessary software libraries:

In [ ]:
# Standard tools for this notebook.
import os                          # general OS helpers
import torch                       # PyTorch: builds the actor and critic networks
import gym                         # OpenAI Gym, provides the Acrobot environment
import numpy as np                 # numerical helper library
import matplotlib.pyplot as plt    # plotting
from tqdm import tqdm              # progress bar shown while training
from torch import nn as nn         # neural-network building blocks
from torch.optim import AdamW      # optimiser that updates the network weights
import torch.nn.functional as F    # functional helpers such as the MSE loss

## Create and preprocess the environment

### Create the environment

In [ ]:
# Create the Acrobot environment: a two-link pendulum that must swing its tip above a line.
env = gym.make('Acrobot-v1')

In [ ]:
# Number of values describing a state (6 for Acrobot).
dims = env.observation_space.shape[0]
# Number of available actions (3: apply torque -1, 0, or +1).
actions = env.action_space.n

print(f"State dimensions: {dims}. Actions: {actions}")
print(f"Sample state: {env.reset()}")

In [ ]:
# Render one frame of the environment as an image.
plt.imshow(env.render(mode='rgb_array'))

### Prepare the environment to work with PyTorch

In [ ]:
# Wrapper that turns Gym's NumPy arrays into PyTorch tensors with the shapes the training loop expects.
class PreprocessEnv(gym.Wrapper):

    def __init__(self, env):
        # Inherit all behaviour from the wrapped environment.
        gym.Wrapper.__init__(self, env)

    def reset(self):
        # Reset and convert the starting state to a float tensor.
        state = self.env.reset()
        return torch.from_numpy(state).float()

    def step(self, actions):
        # Convert the action tensor back to NumPy for Gym.
        actions = actions.squeeze().numpy()
        # Take the action(s) in the environment.
        next_state, reward, done, info = self.env.step(actions)
        # Convert results back to tensors.
        next_state = torch.from_numpy(next_state).float()
        reward = torch.tensor(reward).unsqueeze(1).float()
        done = torch.tensor(done).unsqueeze(1)
        return next_state, reward, done, info

In [ ]:
# Run 8 Acrobot environments in parallel to gather experience faster.
num_envs = 8
# A vectorised environment that steps all copies at once.
parallel_env = gym.vector.make('Acrobot-v1', num_envs=num_envs)
# Seed everything for reproducibility.
seed_everything(parallel_env)
# Wrap it so it returns PyTorch tensors.
parallel_env = PreprocessEnv(parallel_env)

### Create the policy $\pi(s)$ -- the **Actor**

The **actor** is the policy network: given a state it outputs a probability for each action. This is the part that actually *acts* in the world. We will sample actions from it and, just like in REINFORCE, nudge it toward actions that turn out better than expected.

In [ ]:
# The ACTOR: the policy network. It outputs a probability for each action (Softmax).
# The actor decides WHAT TO DO; we sample actions from its distribution.
actor = nn.Sequential(
    nn.Linear(dims, 128),     # input: state (6 numbers) -> 128 hidden units
    nn.ReLU(),                # non-linear activation
    nn.Linear(128, 64),       # hidden layer: 128 -> 64
    nn.ReLU(),                # non-linear activation
    nn.Linear(64, actions),   # output: 64 -> one score per action
    nn.Softmax(dim=-1))       # turn the scores into action probabilities

### Create the value network $v(s)$ -- the **Critic**

The **critic** is a value network: given a state it outputs a single number $v(s)$ estimating how much total reward we expect from that state onward. The critic never chooses actions; its job is to *judge* states and provide a **baseline**. Comparing what actually happened against the critic's prediction gives us the **advantage**, which tells the actor whether an action was better or worse than average.

In [ ]:
# The CRITIC: the value network. It outputs a single number v(s): how good the state is.
# The critic JUDGES states; it provides the baseline the actor's updates are measured against.
critic = nn.Sequential(
    nn.Linear(dims, 128),   # input: state (6 numbers) -> 128 hidden units
    nn.ReLU(),              # non-linear activation
    nn.Linear(128, 64),     # hidden layer: 128 -> 64
    nn.ReLU(),              # non-linear activation
    nn.Linear(64, 1))       # output: a single value estimate for the state

## Implement the algorithm

### Advantage Actor-Critic in plain words

REINFORCE has a weakness: it must wait for a whole episode to finish, and its updates are very *noisy* because the raw return $G_t$ swings wildly. **Actor-Critic** fixes both problems by training **two networks together**:

- The **actor** is the policy -- it decides what to do.
- The **critic** is a value function $v(s)$ -- it estimates how good a state is and acts as a **baseline**.

Instead of the full Monte-Carlo return, A2C uses a **one-step (TD) estimate** of the return:

$$\hat G_t = R_{t+1} + \gamma\, v(S_{t+1}\mid w).$$

From this we form the **advantage**:

$$A_t = \underbrace{R_{t+1} + \gamma\, v(S_{t+1})}_{\text{what actually happened}} \; - \; \underbrace{v(S_t)}_{\text{what the critic expected}}.$$

- If the advantage is **positive**, the action did better than the critic expected, so make it **more** likely.
- If it is **negative**, make it **less** likely.

Each step we do two updates:

1. **Critic update** -- move $v(S_t)$ toward the TD target $\hat G_t$ (mean-squared-error loss).
2. **Actor update** -- push the log-probability of the chosen action up or down in proportion to the advantage.

Subtracting the critic's baseline removes most of the noise that made plain REINFORCE slow, so A2C learns faster and more stably -- and it can update every single step instead of waiting for the episode to end.

In [ ]:
# Advantage Actor-Critic (A2C) training loop.
def actor_critic(actor, critic, episodes, alpha=1e-4, gamma=0.99):
    # Each network gets its own optimiser (and its own learning rate).
    actor_optim = AdamW(actor.parameters(), lr=1e-3)
    critic_optim = AdamW(critic.parameters(), lr=1e-4)
    # Track the actor loss, critic loss, and the return of each episode.
    stats = {'Actor Loss': [], 'Critic Loss': [], 'Returns': []}

    for episode in tqdm(range(1, episodes + 1)):
        # Reset all parallel environments.
        state = parallel_env.reset()
        # Track which environments have finished.
        done_b = torch.zeros((num_envs, 1), dtype=torch.bool)
        ep_return = torch.zeros((num_envs, 1))
        # I discounts the actor update the deeper we are into the episode (gamma**t).
        I = 1.

        while not done_b.all():
            # ----- Actor picks an action by sampling from its probabilities -----
            action = actor(state).multinomial(1).detach()
            # Take the action.
            next_state, reward, done, _ = parallel_env.step(action)

            # ----- Train the CRITIC with a one-step TD target -----
            # The critic's current estimate of this state's value.
            value = critic(state)
            # TD target: reward now + discounted value of the NEXT state (no future value if episode ended).
            # .detach() stops gradients flowing through the target (we bootstrap, not differentiate it).
            target = reward + ~done * gamma * critic(next_state).detach()
            # Critic loss: how far its estimate is from the TD target.
            critic_loss = F.mse_loss(value, target)
            critic.zero_grad()
            critic_loss.backward()
            critic_optim.step()

            # ----- Train the ACTOR using the advantage -----
            # Advantage = how much better this step turned out than the critic expected.
            # Positive advantage -> make this action more likely; negative -> less likely.
            advantage = (target - value).detach()
            # Action probabilities (with gradients this time).
            probs = actor(state)
            log_probs = torch.log(probs + 1e-6)
            # Log-probability of the action we actually took.
            action_log_prob = log_probs.gather(1, action)
            # Entropy bonus to keep the policy exploring.
            entropy = - torch.sum(probs * log_probs, dim=-1, keepdim=True)
            # Policy-gradient loss weighted by the advantage, plus the entropy bonus.
            actor_loss = - I * action_log_prob * advantage - 0.01 * entropy
            # Average over the parallel environments.
            actor_loss = actor_loss.mean()
            actor.zero_grad()
            actor_loss.backward()
            actor_optim.step()

            # Bookkeeping for this step.
            ep_return += reward
            done_b |= done
            state = next_state
            # Increase the discount applied to deeper steps.
            I = I * gamma

        # Record stats for this episode.
        stats['Actor Loss'].append(actor_loss.item())
        stats['Critic Loss'].append(critic_loss.item())
        stats['Returns'].append(ep_return.mean().item())

    return stats

In [ ]:
# Train the actor and critic together for 100 episodes.
stats = actor_critic(actor, critic, 100)

## Show results

### Show execution stats

In [ ]:
# Plot the actor loss, critic loss, and the episode returns.
plot_stats(stats)

### Test the resulting agent

In [ ]:
# Watch the trained actor (policy) play 2 episodes.
test_policy_network(env, actor, episodes=2)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch.13](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)